# 🧠 Aurum Brain AI - Colab Free Tier Training

Training LoRA fine-tuning Qwen2.5 di **Google Colab Free (T4 GPU 16GB)**.

## ⚡ Quick Start
1. **Runtime → Change runtime type → GPU T4**
2. **Runtime → Run all** (atau jalankan cell per cell)
3. Tunggu ~30-60 menit (tergantung model size)
4. Artifact GGUF otomatis upload ke GitHub Release & Hugging Face

## 🔧 Config

In [ ]:
# ============================================
# KONFIGURASI - EDIT SESUAI KEBUTUHAN
# ============================================
CONFIG = {
    # Model base (1.5B lebih cepat & aman untuk free tier, 3B muat tapi lambat)
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",  # atau "Qwen/Qwen2.5-3B-Instruct"
    
    # Training hyperparameters
    "epochs": 2,
    "batch_size": 1,
    "grad_accum": 4,
    "max_len": 768,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    
    # Output
    "output_dir": "/content/aurum-brain-ai/out",
    "gguf_name": "aurum-brain-q4_k_m.gguf",
    
    # GitHub Release (butuh GH_TOKEN secret)
    "github_repo": "aurum-lab/aurum-brain-ai",
    "create_github_release": True,
    
    # Hugging Face Hub (butuh HF_TOKEN secret)
    "hf_repo_id": "arissuga/aurum-brain-ai",
    "push_to_hf": True,
    
    # Supabase Metrics (optional - butuh SUPABASE_URL & SUPABASE_SERVICE_ROLE_KEY)
    "push_supabase": True,
}

In [ ]:
# ============================================
# SECRETS SETUP - JALANKAN CELL INI DULU
# ============================================
# Masukkan secrets di Colab: ⚙️ Settings → Secrets → Add new secret
# Nama secret: GH_TOKEN, HF_TOKEN, SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY

from google.colab import userdata
import os

secrets_map = {
    "GH_TOKEN": "GH_TOKEN",
    "HF_TOKEN": "HF_TOKEN", 
    "SUPABASE_URL": "SUPABASE_URL",
    "SUPABASE_SERVICE_ROLE_KEY": "SUPABASE_SERVICE_ROLE_KEY"
}

for env_name, secret_name in secrets_map.items():
    try:
        value = userdata.get(secret_name)
        os.environ[env_name] = value
        print(f"✅ {env_name} loaded")
    except Exception as e:
        print(f"⚠️ {env_name} not set: {e}")
        os.environ[env_name] = ""

# Set HF token for transformers
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ.get("HF_TOKEN", "")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# ============================================
# INSTALL DEPENDENCIES
# ============================================
!pip install -q -U pip
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers peft datasets accelerate sentencepiece protobuf bitsandbytes
!pip install -q huggingface_hub hf_transfer
!pip install -q requests tqdm

# Install llama.cpp dengan CUDA support
!apt-get update -qq && apt-get install -y -qq build-essential cmake git 2>/dev/null
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp 2>/dev/null || cd /content/llama.cpp && git pull
!cd /content/llama.cpp && cmake -B build -DLLAMA_CUBLAS=ON -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF
!cd /content/llama.cpp && cmake --build build --config Release -j $(nproc) --target llama-quantize
!ln -sf /content/llama.cpp/build/bin/llama-quantize /usr/local/bin/llama-quantize
!ln -sf /content/llama.cpp/build/bin/llama-quantize /usr/local/bin/quantize

print("✅ Dependencies installed")

In [ ]:
# ============================================
# CLONE REPO & SETUP DATASET
# ============================================
import os
import subprocess

REPO_DIR = "/content/aurum-brain-ai"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aurum-lab/aurum-brain-ai.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)

# Build dataset
!python scripts/build_dataset.py
dataset_size = sum(1 for _ in open("data/train.jsonl"))
print(f"📊 Dataset: {dataset_size} samples")

In [ ]:
# ============================================
# TRAINING FUNCTION (adapted dari scripts/train.py)
# ============================================
import os
import json
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from tqdm.auto import tqdm

def train_lora(config):
    base_model = config["base_model"]
    output_dir = config["output_dir"]
    
    print(f"🚀 Loading {base_model}...")
    
    # 4-bit quantization untuk hemat VRAM
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        use_auth_token=os.environ.get("HF_TOKEN") or True
    )
    
    tokenizer = AutoTokenizer.from_pretrained(
        base_model,
        trust_remote_code=True,
        use_auth_token=os.environ.get("HF_TOKEN") or True
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # Prepare for k-bit training
    model = prepare_model_for_kbit_training(model)
    
    # LoRA config
    lora_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Load dataset
    def load_and_tokenize():
        rows = []
        with open("data/train.jsonl", "r") as f:
            for line in f:
                rows.append(json.loads(line))
        
        def format_chat(row):
            msgs = [
                {"role": "system", "content": row["system"]},
                {"role": "user", "content": row["user"]},
                {"role": "assistant", "content": row["assistant"]}
            ]
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        
        texts = [format_chat(r) for r in rows]
        
        def tokenize_fn(examples):
            return tokenizer(
                examples["text"],
                truncation=True,
                max_length=config["max_len"],
                padding="max_length",
                return_tensors="pt"
            )
        
        dataset = Dataset.from_dict({"text": texts})
        dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
        dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
        return dataset
    
    train_dataset = load_and_tokenize()
    
    # Training args
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch_size"],
        gradient_accumulation_steps=config["grad_accum"],
        learning_rate=config["learning_rate"],
        bf16=True,
        logging_steps=5,
        save_strategy="epoch",
        save_total_limit=2,
        remove_unused_columns=False,
        report_to="none",
        optim="adamw_torch",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        gradient_checkpointing=True,
        dataloader_pin_memory=False,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    
    print("🏋️ Starting training...")
    trainer.train()
    
    # Save LoRA adapter
    adapter_dir = Path(output_dir) / "adapter"
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    print(f"✅ LoRA adapter saved to {adapter_dir}")
    
    # Merge & save full model
    print("🔗 Merging LoRA weights...")
    merged_model = model.merge_and_unload()
    merged_dir = Path(output_dir) / "merged"
    merged_model.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    print(f"✅ Merged model saved to {merged_dir}")
    
    return str(merged_dir)

# Run training
merged_path = train_lora(CONFIG)

In [ ]:
# ============================================
# CONVERT TO GGUF (Q4_K_M)
# ============================================
import subprocess
from pathlib import Path

merged_dir = Path(merged_path)
gguf_output = Path(CONFIG["output_dir"]) / CONFIG["gguf_name"]

# Convert HF -> GGUF (F16 dulu)
f16_gguf = Path(CONFIG["output_dir"]) / "aurum-brain-f16.gguf"

print("🔄 Converting to GGUF F16...")
result = subprocess.run([
    "python", "/content/llama.cpp/convert_hf_to_gguf.py",
    str(merged_dir),
    "--outfile", str(f16_gguf),
    "--outtype", "f16"
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ Convert failed: {result.stderr}")
    raise RuntimeError("GGUF conversion failed")
print(f"✅ F16 GGUF: {f16_gguf}")

# Quantize F16 -> Q4_K_M
print("🔄 Quantizing to Q4_K_M...")
result = subprocess.run([
    "/usr/local/bin/llama-quantize",
    str(f16_gguf),
    str(gguf_output),
    "Q4_K_M"
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ Quantize failed: {result.stderr}")
    raise RuntimeError("Quantization failed")

# Cleanup F16
f16_gguf.unlink(missing_ok=True)

gguf_size_mb = gguf_output.stat().st_size / 1024 / 1024
print(f"✅ GGUF Q4_K_M ready: {gguf_output} ({gguf_size_mb:.1f} MB)")

# Verify GGUF
import struct
with open(gguf_output, 'rb') as f:
    magic = f.read(4)
    assert magic == b'GGUF', 'Invalid GGUF'
    version = struct.unpack('<I', f.read(4))[0]
    print(f"✅ GGUF valid: magic={magic}, version={version}")

In [ ]:
# ============================================
# UPLOAD TO GITHUB RELEASE
# ============================================
import subprocess
import os
from datetime import datetime

if CONFIG["create_github_release"] and os.environ.get("GH_TOKEN"):
    repo = CONFIG["github_repo"]
    ver = f"v{datetime.utcnow().strftime('%Y.%m.%d')}-colab-{os.environ.get('GITHUB_SHA', 'manual')[:7]}"
    dataset_size = sum(1 for _ in open("data/train.jsonl"))
    
    # Create release notes
    notes = f"""## Aurum Brain AI - Colab Free Tier Training

Model fine-tuned di Google Colab Free (T4 GPU).

### Specs
- Base: `{CONFIG['base_model']}`
- LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}
- Dataset: {dataset_size} samples
- Quantization: Q4_K_M
- Training: {CONFIG['epochs']} epochs, max_len={CONFIG['max_len']}

### Usage
**PocketPal**: Download GGUF → Import → set system prompt

**Ollama**:
```bash
ollama create aurum-brain -f modelfile
ollama run aurum-brain
```

**llama.cpp**:
```bash
./main -m aurum-brain-q4_k_m.gguf -p 'Halo!'
```
"""
    
    notes_file = "/tmp/release_notes.md"
    with open(notes_file, "w") as f:
        f.write(notes)
    
    gguf_path = Path(CONFIG["output_dir"]) / CONFIG["gguf_name"]
    system_prompt = "data/system_prompt.txt"
    
    env = os.environ.copy()
    env["GH_TOKEN"] = os.environ["GH_TOKEN"]
    
    result = subprocess.run([
        "gh", "release", "create", ver,
        str(gguf_path), system_prompt,
        "--repo", repo,
        "--title", f"Aurum Brain AI {ver}",
        "--notes-file", notes_file
    ], env=env, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✅ GitHub Release created: {ver}")
    else:
        print(f"⚠️ GitHub Release failed: {result.stderr}")
else:
    print("⏭️ Skip GitHub Release (no GH_TOKEN or disabled)")

In [ ]:
# ============================================
# UPLOAD TO HUGGING FACE HUB
# ============================================
import os
from pathlib import Path
from huggingface_hub import HfApi

if CONFIG["push_to_hf"] and os.environ.get("HF_TOKEN"):
    api = HfApi(token=os.environ["HF_TOKEN"])
    repo_id = CONFIG["hf_repo_id"]
    
    gguf_path = Path(CONFIG["output_dir"]) / CONFIG["gguf_name"]
    system_prompt = Path("data/system_prompt.txt")
    
    print(f"📤 Uploading to {repo_id}...")
    
    # Upload GGUF
    api.upload_file(
        path_or_fileobj=str(gguf_path),
        path_in_repo=CONFIG["gguf_name"],
        repo_id=repo_id,
        repo_type="model"
    )
    
    # Upload system prompt
    api.upload_file(
        path_or_fileobj=str(system_prompt),
        path_in_repo="system_prompt.txt",
        repo_id=repo_id,
        repo_type="model"
    )
    
    print(f"✅ Uploaded to https://huggingface.co/{repo_id}")
else:
    print("⏭️ Skip HF Hub (no HF_TOKEN or disabled)")

In [ ]:
# ============================================
# PUSH METRICS TO SUPABASE
# ============================================
import os
import json
import requests
from datetime import datetime
from pathlib import Path

if CONFIG["push_supabase"] and os.environ.get("SUPABASE_URL") and os.environ.get("SUPABASE_SERVICE_ROLE_KEY"):
    url = os.environ["SUPABASE_URL"]
    key = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
    
    headers = {
        "apikey": key,
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json",
        "Prefer": "return=representation"
    }
    
    def sb_request(method, table, data=None, params=None):
        resp = requests.request(method, f"{url}/rest/v1/{table}", headers=headers, json=data, params=params, timeout=30)
        resp.raise_for_status()
        return resp.json() if resp.text else None
    
    # Get current BrainStat
    try:
        current = sb_request("GET", "BrainStat", params={"id": "eq.default", "select": "*"})
        current = current[0] if current else {}
    except:
        current = {}
    
    dataset_size = sum(1 for _ in open("data/train.jsonl"))
    gguf_size_mb = round(Path(CONFIG["output_dir"]) / CONFIG["gguf_name"].stat().st_size / 1024 / 1024, 1)
    total_mem = current.get("totalMemories", 0) + dataset_size
    maturity = min(10.0, total_mem / 10000)
    
    # Upsert BrainStat
    brainstat = {
        "id": "default",
        "totalMemories": total_mem,
        "totalSignals": current.get("totalSignals", 0),
        "totalWinSignals": current.get("totalWinSignals", 0),
        "totalLossSignals": current.get("totalLossSignals", 0),
        "winRate": current.get("winRate", 0),
        "avgConfidence": current.get("avgConfidence", 0),
        "totalPnl": current.get("totalPnl", 0),
        "learningStreak": current.get("learningStreak", 0) + 1,
        "lastLearnAt": datetime.utcnow().isoformat() + "Z",
        "maturity": round(maturity, 2),
        "updatedAt": datetime.utcnow().isoformat() + "Z"
    }
    sb_request("POST", "BrainStat", data=brainstat, params={"on_conflict": "id"})
    
    # Insert LearningLog
    learninglog = {
        "sessionDate": datetime.utcnow().isoformat() + "Z",
        "trigger": "Google Colab Free Tier",
        "memoriesCreated": dataset_size,
        "memoriesUpdated": 0,
        "patternsFound": 0,
        "signalsAnalyzed": 0,
        "avgConfidenceBefore": current.get("avgConfidence", 0),
        "avgConfidenceAfter": current.get("avgConfidence", 0),
        "weightDeltaTotal": 0,
        "summary": f"LoRA fine-tune on {CONFIG['base_model']}, {CONFIG['epochs']} epochs, {dataset_size} samples, GGUF {gguf_size_mb}MB",
        "details": json.dumps({
            "base_model": CONFIG["base_model"],
            "epochs": CONFIG["epochs"],
            "dataset_size": dataset_size,
            "gguf_size_mb": gguf_size_mb,
            "lora_r": CONFIG["lora_r"],
            "lora_alpha": CONFIG["lora_alpha"],
            "max_len": CONFIG["max_len"]
        }),
        "duration": 0,
        "createdAt": datetime.utcnow().isoformat() + "Z"
    }
    sb_request("POST", "LearningLog", data=learninglog)
    
    print("✅ Supabase metrics pushed")
else:
    print("⏭️ Skip Supabase (secrets not set or disabled)")

In [ ]:
# ============================================
# DOWNLOAD ARTIFACTS (optional - untuk backup lokal)
# ============================================
from google.colab import files
from pathlib import Path

gguf_path = Path(CONFIG["output_dir"]) / CONFIG["gguf_name"]
system_prompt = Path("data/system_prompt.txt")

print(f"📥 Downloading {gguf_path.name} ({gguf_path.stat().st_size/1024/1024:.1f} MB)...")
files.download(str(gguf_path))
files.download(str(system_prompt))

print("\n✅ ALL DONE!")
print(f"Model: {CONFIG['base_model']}")
print(f"GGUF: {CONFIG['gguf_name']}")
if CONFIG["create_github_release"]:
    print(f"GitHub: https://github.com/{CONFIG['github_repo']}/releases")
if CONFIG["push_to_hf"]:
    print(f"HF Hub: https://huggingface.co/{CONFIG['hf_repo_id']}")